In [ ]:
import pandas as pd
import ast
import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env file in project root
project_root = Path(__file__).parent.parent.parent if '__file__' in locals() else Path.cwd().parent.parent
dotenv_path = project_root / '.env'
load_dotenv(dotenv_path=dotenv_path) 

PAPER_PATH = Path(os.getenv('PAPER_PATH'))
print(PAPER_PATH)


In [ ]:
# Load verified tuples from US and global symlinked datasets
verify_path_us = '/share/pierson/matt/UAIR/outputs/2025-11-12/domain_agnostic_ai_us/full_event_pipeline_general_ai/outputs/verify_nbl/verify_nbl_results.parquet'
verify_path_global = '/share/pierson/matt/UAIR/outputs/2025-11-13/domain_agnostic_ai/full_event_pipeline_general_ai/outputs/verify_nbl/verify_nbl_results.parquet'

df_verify_us = pd.read_parquet(verify_path_us)
df_verify_global = pd.read_parquet(verify_path_global)

# Combine US and global datasets
df_verify_all = pd.concat([df_verify_us, df_verify_global], ignore_index=True)

# Filter to only verified tuples
decomposed_articles = df_verify_all[df_verify_all['core_tuple_verified'] == True].copy()

print(f'Total verified tuples: {len(decomposed_articles):,} (US: {len(df_verify_us[df_verify_us["core_tuple_verified"] == True]):,}, Global: {len(df_verify_global[df_verify_global["core_tuple_verified"] == True]):,})')


In [ ]:
decomposed_articles.columns

In [ ]:
len(decomposed_articles)

In [ ]:
decomposed_articles['deployment_domain'].value_counts()

In [ ]:
def flatten_string_lists(series):
    """Convert string representations of lists to actual lists, then flatten into one list"""
    all_items = []
    for item in series:
        # Convert string representation to actual list
        if isinstance(item, str):
            parsed = ast.literal_eval(item)
        else:
            parsed = item
        # Extend the all_items list with the parsed list
        if isinstance(parsed, list):
            all_items.extend(parsed)
    
    # Filter out empty elements (empty strings, None, whitespace-only strings)
    all_items = [item for item in all_items if item and str(item).strip()]
    
    return all_items

decomposed_articles_by_domain = decomposed_articles.groupby('deployment_domain')
# for each domain, aggregate via concatenation to get ALL risks, harms, and benefits mentioned in the group. each row is a list, so we need to flatten the lists into one list per domain
decomposed_articles_by_domain = decomposed_articles_by_domain.agg(
    num_articles=('article_id', 'nunique'),
    risks=('list_of_risks_that_occurred', flatten_string_lists),  
    harms=('list_of_harms_that_occurred', flatten_string_lists),
    benefits=('list_of_benefits_that_occurred', flatten_string_lists)

)

# drop empty lists and reset index 
decomposed_articles_by_domain = decomposed_articles_by_domain.dropna()
decomposed_articles_by_domain = decomposed_articles_by_domain.reset_index()

decomposed_articles_by_domain.sort_values(by='num_articles', ascending=False, inplace=True)







In [ ]:
decomposed_articles_by_domain.head(n=10)

In [ ]:
# make a latex table of the top 10 domains, showing number of articles, and the risks, harms, and benefits 

# Configuration parameters
N_ITEMS_PER_CATEGORY = 6  # Number of risks/harms/benefits to show per domain
N_TOP_DOMAINS = 10  # Number of top domains to include

top_10_domains = decomposed_articles_by_domain.head(n=N_TOP_DOMAINS).copy()

def format_list_for_latex(items, max_items=None):
    """Format a list of items as a LaTeX itemize environment"""
    # Filter out empty elements (empty strings, None, whitespace-only strings)
    if items:
        items = [item for item in items if item and str(item).strip()]
    
    if not items or len(items) == 0:
        return "None reported"
    
    # Limit items if specified
    total_items = len(items)
    items_to_show = items[:max_items] if max_items else items
    is_truncated = max_items and total_items > max_items
    
    # Escape special LaTeX characters
    def escape_latex(text):
        text = str(text)
        # First, unescape any existing LaTeX escapes to avoid double-escaping
        text = text.replace(r'\$', '$').replace(r'\_', '_').replace(r'\&', '&').replace(r'\%', '%').replace(r'\#', '#')
        
        replacements = {
            '&': r'\&',
            '%': r'\%',
            '$': r'\$',
            '#': r'\#',
            '_': r'\_',
            '{': r'\{',
            '}': r'\}',
            '~': r'\textasciitilde{}',
            '^': r'\^{}',
        }
        for old, new in replacements.items():
            text = text.replace(old, new)
        return text
    
    # Create itemize list with smaller font
    escaped_items = [escape_latex(item) for item in items_to_show]
    itemized = r'\begin{itemize}[leftmargin=*, nosep, before=\vspace{-0.5\baselineskip}, after=\vspace{-0.5\baselineskip}] ' + '\n'
    itemized += r'\footnotesize' + '\n'
    for item in escaped_items:
        itemized += r'\item ' + item + '\n'
    
    # Add truncation note if applicable
    if is_truncated:
        itemized += r'\item \textit{[...and ' + str(total_items - max_items) + r' more]}' + '\n'
    
    itemized += r'\end{itemize}'
    
    return itemized

# Build custom LaTeX table manually for better formatting
def build_custom_latex_table(domains_df, n_items):
    """Build a custom LaTeX table with domain headers spanning all columns"""
    
    # Start the longtable environment
    latex = r'\begin{longtable}{p{5.2cm}|p{5.2cm}|p{5.2cm}}' + '\n'
    latex += r'\caption{Top ' + str(len(domains_df)) + r' AI Deployment Domains by Article Count with Reported Risks, Harms, and Benefits}' + '\n'
    latex += r'\label{tab:top_domains_detailed}\\' + '\n'
    
    # First head - empty, as we'll add domain-specific headers
    latex += r'\endfirsthead' + '\n\n'
    
    # Continued header for subsequent pages - also empty
    latex += r'\endhead' + '\n\n'
    
    # Footer for non-final pages - empty
    latex += r'\endfoot' + '\n\n'
    
    # Final footer
    latex += r'\endlastfoot' + '\n\n'
    
    # Add each domain
    for idx, row in domains_df.iterrows():
        domain = row['deployment_domain']
        n_articles = row['num_articles']
        risks = row['risks']
        harms = row['harms']
        benefits = row['benefits']
        
        # Domain header spanning all columns
        latex += r'\multicolumn{3}{l}{\textbf{\large ' + escape_latex(domain) + r'} \textit{(N=' + str(n_articles) + r' articles)}} \\' + '\n'
        latex += r'\midrule' + '\n'
        
        # Column headers for this domain
        latex += r'\textbf{Risks} & \textbf{Harms} & \textbf{Benefits} \\' + '\n'
        latex += r'\midrule' + '\n'
        
        # Format the three columns
        risks_text = format_list_for_latex(risks, max_items=n_items)
        harms_text = format_list_for_latex(harms, max_items=n_items)
        benefits_text = format_list_for_latex(benefits, max_items=n_items)
        
        # Add content row
        latex += risks_text + ' & ' + harms_text + ' & ' + benefits_text + r' \\' + '\n'
        latex += r'\pagebreak' + '\n'
    
    # End the table
    latex += r'\end{longtable}' + '\n'
    
    return latex

def escape_latex(text):
    """Escape special LaTeX characters"""
    text = str(text)
    # First, unescape any existing LaTeX escapes to avoid double-escaping
    text = text.replace(r'\$', '$').replace(r'\_', '_').replace(r'\&', '&').replace(r'\%', '%').replace(r'\#', '#')
    
    replacements = {
        '&': r'\&',
        '%': r'\%',
        '$': r'\$',
        '#': r'\#',
        '_': r'\_',
        '{': r'\{',
        '}': r'\}',
        '~': r'\textasciitilde{}',
        '^': r'\^{}',
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text

# Generate the custom table
latex_table = build_custom_latex_table(top_10_domains, N_ITEMS_PER_CATEGORY)

# Print the LaTeX table code
print("% Required packages in your main document preamble:")
print("% \\usepackage{longtable}")
print("% \\usepackage{booktabs}")
print("% \\usepackage{enumitem}")
print("% \\usepackage{array}")
print()
print("% Table code (ready for \\input):")
print()
print(latex_table)

# write to .tex file in papers directory / tables 
with open(PAPER_PATH / 'tables' / 'top_domains_detailed.tex', 'w') as f:
    f.write(latex_table)



In [ ]:
# Build and save a simple 2-column LaTeX table of top 10 domains by article count

N_TOP_DOMAINS_SIMPLE = 10

# Ensure we have the required DataFrame and it is sorted by num_articles desc
assert 'decomposed_articles_by_domain' in globals(), "Expected DataFrame 'decomposed_articles_by_domain' to exist"

# Select only required columns
_top = decomposed_articles_by_domain[['deployment_domain', 'num_articles']].head(n=N_TOP_DOMAINS_SIMPLE).copy()

# Escape LaTeX helper (redefine here to make this cell standalone)
def escape_latex(text):
    text = str(text)
    text = text.replace(r'\$', '$').replace(r'\_', '_').replace(r'\&', '&').replace(r'\%', '%').replace(r'\#', '#')
    replacements = {
        '&': r'\&',
        '%': r'\%',
        '$': r'\$',
        '#': r'\#',
        '_': r'\_',
        '{': r'\{',
        '}': r'\}',
        '~': r'\textasciitilde{}',
        '^': r'\^{}',
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text

# Construct LaTeX longtable
latex_lines = []
latex_lines.append(r"\begin{longtable}{p{9cm}r}")
latex_lines.append(r"\caption{Top 10 AI Deployment Domains by Article Count}")
latex_lines.append(r"\label{tab:top_domains_by_article_count}\\")
latex_lines.append(r"\toprule")
latex_lines.append(r"\textbf{Deployment domain} & \textbf{Num. articles} \\")
latex_lines.append(r"\midrule")
latex_lines.append(r"\endfirsthead")
latex_lines.append(r"\toprule")
latex_lines.append(r"\textbf{Deployment domain} & \textbf{Num. articles} \\")
latex_lines.append(r"\midrule")
latex_lines.append(r"\endhead")
latex_lines.append(r"\endfoot")
latex_lines.append(r"\endlastfoot")

for _, row in _top.iterrows():
    domain = escape_latex(row['deployment_domain'])
    count = int(row['num_articles'])
    latex_lines.append(f"{domain} & {count} " + r"\\")

latex_lines.append(r"\bottomrule")
latex_lines.append(r"\end{longtable}")

latex_table_simple = "\n".join(latex_lines)

print("% Required packages in your main document preamble:")
print("% \\usepackage{longtable}")
print("% \\usepackage{booktabs}")
print("% \\usepackage{array}")
print()
print("% Table code (ready for \\input):")
print()
print(latex_table_simple)

# Save to PAPER_PATH / tables
output_path = PAPER_PATH / 'tables' / 'top_domains_by_article_count.tex'
with open(output_path, 'w') as f:
    f.write(latex_table_simple)
print(f"Saved to: {output_path}")


In [ ]:
# Simple 2-column LaTeX table: Top 10 domains by article count (proper backslashes)

N_TOP_DOMAINS_SIMPLE = 10

assert 'decomposed_articles_by_domain' in globals(), "Expected DataFrame 'decomposed_articles_by_domain' to exist"

_top = decomposed_articles_by_domain[['deployment_domain', 'num_articles']].head(n=N_TOP_DOMAINS_SIMPLE).copy()

# Minimal LaTeX escaping
def escape_latex(text):
    text = str(text)
    text = text.replace(r'\$', '$').replace(r'\_', '_').replace(r'\&', '&').replace(r'\%', '%').replace(r'\#', '#')
    for old, new in {
        '&': r'\&',
        '%': r'\%',
        '$': r'\$',
        '#': r'\#',
        '_': r'\_',
        '{': r'\{',
        '}': r'\}',
        '~': r'\textasciitilde{}',
        '^': r'\^{}',
    }.items():
        text = text.replace(old, new)
    return text

latex_lines = []
latex_lines.append(r"\begin{longtable}{p{9cm}r}")
latex_lines.append(r"\caption{Top 10 AI Deployment Domains by Article Count}")
latex_lines.append(r"\label{tab:top_domains_by_article_count}\\")
latex_lines.append(r"\toprule")
latex_lines.append(r"\textbf{Deployment domain} & \textbf{Num. articles} \\")
latex_lines.append(r"\midrule")
latex_lines.append(r"\endfirsthead")
latex_lines.append(r"\toprule")
latex_lines.append(r"\textbf{Deployment domain} & \textbf{Num. articles} \\")
latex_lines.append(r"\midrule")
latex_lines.append(r"\endhead")
latex_lines.append(r"\endfoot")
latex_lines.append(r"\endlastfoot")

for _, row in _top.iterrows():
    domain = escape_latex(row['deployment_domain'])
    count = int(row['num_articles'])
    latex_lines.append(f"{domain} & {count} " + r"\\")

latex_lines.append(r"\bottomrule")
latex_lines.append(r"\end{longtable}")

latex_table_simple = "\n".join(latex_lines)

print("% Required packages in your main document preamble:")
print("% \\usepackage{longtable}")
print("% \\usepackage{booktabs}")
print("% \\usepackage{array}")
print()
print("% Table code (ready for \\input):")
print()
print(latex_table_simple)

output_path = PAPER_PATH / 'tables' / 'top_domains_by_article_count.tex'
with open(output_path, 'w') as f:
    f.write(latex_table_simple)
print(f"Saved to: {output_path}")
